In [79]:
import json
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm
from datasets import Dataset
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import shap
import pickle
import string
from collections import defaultdict
import numpy as np
from glob import glob

In [46]:
import logging
logging.getLogger('shap').setLevel(logging.WARNING) # turns off the "shap INFO" logs
logging.getLogger('matplotlib').setLevel(logging.WARNING) # turns off the progress bar

In [47]:
import warnings
warnings.filterwarnings("ignore")

In [48]:
with open("C:\\Users\\imruh\\Documents\\RiskPerceptionSeafare\\params.json", 'r') as f:
    PARAMS = json.load(f)

In [49]:
def clean_text(text):
    # remove tags from xml
    text = re.sub(r"<.*>", ' ', text)
    # remove indication of beginning of paragraph
    text = re.sub(r"^§ [\.\w]*\s*", ' ', text)
    # remove anything in [] 
    text = re.sub(r"\[.*\]", '', text)
    return text.lstrip().rstrip()


In [50]:
def load_dataset():
    dataset_path = f'{PARAMS["roberta_data_path"]}_{PARAMS["word_window"]}_filtered'
    all_excerpts = Dataset.load_from_disk(dataset_path).to_pandas()
    dataset = pd.DataFrame({"text":all_excerpts["text"], "label":all_excerpts["label"]}).dropna().drop_duplicates()
    print("Cleaning text")
    dataset["text"] = [clean_text(x) for x in tqdm(dataset["text"])]

    return dataset

In [51]:
def split_dataset(dataset, labels, train_size, val_size):
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for i, label in enumerate(labels)}
    print(label2id)  
    print(id2label)
    train_data, test_data = train_test_split(dataset, train_size=train_size, stratify=dataset["label"], random_state=42)
    test_data, val_data = train_test_split(test_data, train_size=val_size, stratify=test_data["label"], random_state=42)
    train_data["label"] = [label2id[x] for x in train_data["label"]]
    test_data["label"] = [label2id[x] for x in test_data["label"]]
    val_data["label"] = [label2id[x] for x in val_data["label"]]
    train_data = Dataset.from_pandas(train_data).remove_columns(["__index_level_0__"])
    test_data = Dataset.from_pandas(test_data).remove_columns(["__index_level_0__"])
    val_data = Dataset.from_pandas(val_data).remove_columns(["__index_level_0__"])
    
    return label2id, id2label, train_data, test_data, val_data

In [52]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

Cleaning text


100%|██████████| 5859/5859 [00:00<00:00, 202040.82it/s]

Size of dataset: 5859
{'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
{0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
Train: Counter({1: 2089, 2: 1662, 0: 936})
Test: Counter({1: 261, 2: 208, 0: 117})
Val: Counter({1: 261, 2: 208, 0: 117})


In [53]:
dataset["label"].value_counts()

label
MEDIUM    2611
LOW       2078
HIGH      1170
Name: count, dtype: int64

In [54]:
val_data.to_pandas().to_csv("val_data.csv")

In [55]:
# Load your fine-tuned BERT model and tokenizer
model_path = f'{PARAMS["save_model"]}{model_id.split("/")[-1]}_finetuned_16_12'
model_base = PARAMS["classi_finetune_model"]
tokenizer = AutoTokenizer.from_pretrained(model_base)
# model = AutoModelForSequenceClassification.from_pretrained(model_path)
# model = model.to("cuda")
# model.eval()

In [56]:
# preds = pipeline(
#     "text-classification",
#     model=model,
#     tokenizer=tokenizer,
#     return_all_scores=True,
# )


In [57]:
# explainer = shap.Explainer(
#     preds, seed=42
# )

In [58]:
texts = val_data["text"]

In [59]:
size = 10
# shap_values = []
# for itr in range(0, len(texts), size):
#     print(f"Processing batch {itr} - {itr+size}")
#     batch_texts = texts[itr:itr+size]
#     batch_shap_values = explainer(batch_texts)
#     with open(f"shap_batch_{itr}.pkl", "wb") as f:
#         pickle.dump(batch_shap_values, f)
#     shap_values.append(batch_shap_values)

In [94]:
def get_sorted_shap_files(path_pattern="shap_values/shap_batch_*.pkl"):
    files = glob(path_pattern)

    def extract_number(f):
        match = re.search(r"shap_batch_(\d+)\.pkl", f)
        return int(match.group(1)) if match else -1

    return sorted(files, key=extract_number)

In [95]:
files = get_sorted_shap_files()

In [96]:
files

['shap_values\\shap_batch_0.pkl',
 'shap_values\\shap_batch_10.pkl',
 'shap_values\\shap_batch_20.pkl',
 'shap_values\\shap_batch_30.pkl',
 'shap_values\\shap_batch_40.pkl',
 'shap_values\\shap_batch_50.pkl',
 'shap_values\\shap_batch_60.pkl',
 'shap_values\\shap_batch_70.pkl',
 'shap_values\\shap_batch_80.pkl',
 'shap_values\\shap_batch_90.pkl',
 'shap_values\\shap_batch_100.pkl',
 'shap_values\\shap_batch_110.pkl',
 'shap_values\\shap_batch_120.pkl',
 'shap_values\\shap_batch_130.pkl',
 'shap_values\\shap_batch_140.pkl',
 'shap_values\\shap_batch_150.pkl',
 'shap_values\\shap_batch_160.pkl',
 'shap_values\\shap_batch_170.pkl',
 'shap_values\\shap_batch_180.pkl',
 'shap_values\\shap_batch_190.pkl',
 'shap_values\\shap_batch_200.pkl',
 'shap_values\\shap_batch_210.pkl',
 'shap_values\\shap_batch_220.pkl',
 'shap_values\\shap_batch_230.pkl',
 'shap_values\\shap_batch_240.pkl',
 'shap_values\\shap_batch_250.pkl',
 'shap_values\\shap_batch_260.pkl',
 'shap_values\\shap_batch_270.pkl',
 's

In [116]:
all_class_shap = []
for i in files:
    with open(i, "rb") as f:
        batch_sv = pickle.load(f)
    all_class_shap.extend(batch_sv.values)
            
all_data = []

for itr in range(0, len(texts), size):
    batch_texts = texts[itr:itr+size]
    for x in batch_texts:
        all_data.append(tokenizer.tokenize(x))


In [117]:
len(all_class_shap), len(all_data), len(texts)

(586, 586, 586)

In [119]:
def merge_tokens(tokens, values):
    merged_tokens = []
    merged_values = []

    current_token = ""
    current_value = None

    for t, v in zip(tokens, values):

        # RoBERTa word start token
        if t.startswith("▁") and t not in string.punctuation:

            # flush previous token
            if current_token != "":
                merged_tokens.append(current_token)
                merged_values.append(current_value)

            current_token = t[1:]  # remove _
            current_value = v.copy()

        else:
            if t not in string.punctuation:
                # continuation of same word
                current_token += t
                current_value += v

    # flush last token
    if current_token:
        merged_tokens.append(current_token)
        merged_values.append(current_value)

    return merged_tokens, np.array(merged_values)

In [121]:
class_contribs = [defaultdict(list) for _ in range(3)]

for d, sv in zip(all_data, all_class_shap):
    merged_tokens, merged_values = merge_tokens(d, sv)

    for i, token in enumerate(merged_tokens):
        for c in range(3):
            class_contribs[c][token].append(merged_values[i, c])

In [122]:
agg = []

for c in range(3):
    token_scores = {}
    for token, vals in class_contribs[c].items():
        token_scores[token] = np.mean(vals)  # or sum(vals)

    agg.append(token_scores)
    
top_k = 10

# maybe look at bigrams?
for c in range(3):
    print(f"\nTop words for class {id2label[c]}:")

    # positive contribution
    sorted_tokens = sorted(
            [(t, s) for t, s in agg[c].items() if s > 0],
            key=lambda x: x[1],
            reverse=True
        )
    
    for token, score in sorted_tokens[:top_k]:
        print(f"{token}: {score:.4f}")


Top words for class HIGH:
Μασσαλία: 0.2282
(Corsica).: 0.2281
Agrippinus: 0.2048
Isocrates: 0.1847
Charibael: 0.1819
boys: 0.1711
hermaphrodites: 0.1379
Deuso: 0.1235
Gennesareth: 0.1231
Delphians: 0.1216

Top words for class MEDIUM:
consul.,: 0.4438
Carnutes: 0.3381
hermaphrodites: 0.2857
Phrygia: 0.2831
Hyrcanus: 0.2552
Etrusci: 0.2408
Anacreon: 0.2401
Caracalla: 0.2269
(Milion: 0.2258
Livia: 0.2182

Top words for class LOW:
attempted: 0.5136
boys: 0.4955
Alexandria),: 0.3672
Phocaea: 0.3046
Alphins: 0.2983
Thucydides: 0.2943
Heraclas: 0.2777
Antium: 0.2728
Idumaea: 0.2727
Pompei: 0.2705


In [123]:
agg = []

for c in range(3):
    token_scores = {}
    for token, vals in class_contribs[c].items():
        token_scores[token] = np.mean(vals)  # or sum(vals)

    agg.append(token_scores)
    
top_k = 10

for c in range(3):
    print(f"\nTop words for class {id2label[c]}:")

    # negative contribution
    sorted_tokens = sorted(
            [(t, s) for t, s in agg[c].items() if s < 0],
            key=lambda x: x[1],
            reverse=False
        )
    
    for token, score in sorted_tokens[:top_k]:
        print(f"{token}: {score:.4f}")


Top words for class HIGH:
Ionia: -0.0582
Minor: -0.0544
consul.,: -0.0456
Cherronesos: -0.0432
Peter: -0.0390
XIX: -0.0370
Stratos: -0.0317
43°15: -0.0309
Tauroeis: -0.0280
Climachias: -0.0241

Top words for class MEDIUM:
boys: -0.6295
attempted: -0.6027
Deuso: -0.3284
Pompei: -0.3258
Phocaea: -0.3195
Gennesareth: -0.3166
Alexandria),: -0.3080
BABYLON: -0.2848
Phocaeans: -0.2716
Phocaean: -0.2678

Top words for class LOW:
hermaphrodites: -0.4384
Carnutes: -0.3620
Phrygia: -0.2793
sedition: -0.2634
consul.,: -0.2621
Agrippinus: -0.2444
roadstead: -0.2108
pontifex: -0.2074
Isocrates: -0.2045
Parthians: -0.2013
